# Camada Bronze — Ingestão Bruta

Este notebook é responsável pela **primeira camada** da Arquitetura Medalhão.
Aqui os dados são ingeridos **sem transformações**, exatamente como vieram das fontes originais.

Fontes:
- 9 arquivos CSV do dataset Olist (carregados no Volume `medalhao.default.landing`)
- API PTAX do Banco Central do Brasil (cotação do dólar)

Todas as tabelas recebem a coluna `timestamp_ingestion` com o momento exato da carga.

## 0 — Parâmetros do Notebook (Widgets)
As datas de início e fim controlam o período buscado na API PTAX.
Formato esperado pela API: `MM-DD-YYYY`

In [0]:

dbutils.widgets.text("data_inicio", "01-01-2017", "Data Início (MM-DD-YYYY)")
dbutils.widgets.text("data_fim",    "12-31-2018", "Data Fim (MM-DD-YYYY)")

data_inicio = dbutils.widgets.get("data_inicio")
data_fim    = dbutils.widgets.get("data_fim")

print(f"Período selecionado: {data_inicio} → {data_fim}")

## 1 — Imports e Configurações

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
import requests

CATALOG      = "medalhao"
SCHEMA_BRONZE = "bronze"
LANDING_PATH  = f"/Volumes/{CATALOG}/default/landing"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA_BRONZE}")

print("Catálogo e schema configurados:", CATALOG, ">", SCHEMA_BRONZE)

## 2 — Função Auxiliar de Ingestão CSV
Centraliza a leitura dos CSVs e a escrita como tabela Delta na camada Bronze,
adicionando automaticamente a coluna `timestamp_ingestion`.

In [0]:
def ingerir_csv_para_bronze(nome_arquivo: str, nome_tabela: str):
    """
    Lê um CSV do Volume landing e grava como tabela Delta na camada Bronze.
    
    Parâmetros:
        nome_arquivo : nome do arquivo CSV (sem o caminho)
        nome_tabela  : nome da tabela de destino no schema bronze
    """
    caminho = f"{LANDING_PATH}/{nome_arquivo}"
    
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")    
        .option("encoding", "UTF-8")
        .csv(caminho)
    )
    
    df = df.withColumn("timestamp_ingestion", F.current_timestamp())
    
    tabela_completa = f"{CATALOG}.{SCHEMA_BRONZE}.{nome_tabela}"
    
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")  
        .saveAsTable(tabela_completa)
    )
    
    count = spark.table(tabela_completa).count()
    print(f"✅ {tabela_completa} → {count:,} registros ingeridos")

## 3 — Ingestão dos CSVs do Olist
Cada arquivo CSV é mapeado para sua respectiva tabela bronze conforme especificação.

In [0]:

arquivos_tabelas = [
    ("olist_customers_dataset.csv",              "tb_customers"),
    ("olist_geolocation_dataset.csv",            "tb_geolocalizacao"),
    ("olist_order_items_dataset.csv",            "tb_order_items"),
    ("olist_order_payments_dataset.csv",         "tb_order_payments"),
    ("olist_order_reviews_dataset.csv",          "tb_order_reviews"),
    ("olist_orders_dataset.csv",                 "tb_orders"),
    ("olist_products_dataset.csv",               "tb_products"),
    ("olist_sellers_dataset.csv",                "tb_sellers"),
    ("product_category_name_translation.csv",    "tb_product_category_name_translation"),
]

for arquivo, tabela in arquivos_tabelas:
    ingerir_csv_para_bronze(arquivo, tabela)

## 4 — Ingestão via API: Cotação do Dólar (PTAX — Banco Central)

A API PTAX não retorna dados para finais de semana e feriados.
Na camada Bronze salvamos os dados brutos como retornados pela API.
O preenchimento de lacunas (weekends) será feito na Silver.

In [0]:

url_ptax = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    f"CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio}'"
    f"&@dataFinalCotacao='{data_fim}'"
    f"&$select=dataHoraCotacao,cotacaoCompra"
    f"&$format=json"
)

print("Consultando API PTAX...")
response = requests.get(url_ptax, timeout=60)
response.raise_for_status() 

dados_ptax = response.json().get("value", [])
print(f"Registros recebidos da API: {len(dados_ptax)}")

In [0]:

schema_cotacao = StructType([
    StructField("dataHoraCotacao",  StringType(), True),
    StructField("cotacaoCompra",    DoubleType(), True),
])

df_cotacao = spark.createDataFrame(dados_ptax, schema=schema_cotacao)

df_cotacao = df_cotacao.withColumn("timestamp_ingestion", F.current_timestamp())

tabela_cotacao = f"{CATALOG}.{SCHEMA_BRONZE}.tb_cotacao_dolar"

(
    df_cotacao.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_cotacao)
)

count = spark.table(tabela_cotacao).count()
print(f"✅ {tabela_cotacao} → {count:,} registros ingeridos")

## 5 — Validação Final
Exibe todas as tabelas criadas no schema bronze.

In [0]:
print("=== Tabelas na camada Bronze ===")
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA_BRONZE}"))